In [ ]:
import requests
import shutil

url = "http://dataverse.jpl.nasa.gov/api/access/datafile/83039"
output_path = "file.zip"

# Download the file
response = requests.get(url, stream=True)
with open(output_path, 'wb') as output_file:
    shutil.copyfileobj(response.raw, output_file)

# Verify if the download was successful
if response.status_code == 200:
    print("Download successful")
else:
    print(f"Download failed with status code: {response.status_code}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip '/content/drive/MyDrive/caption generator.zip'

# Dont Run above code if you want to run locally

In [ ]:
import os
import os.path as op
from pathlib import Path
import shutil
import logging
import numpy as np
from tqdm import tqdm
from skimage import io
import subprocess
import tensorflow as tf
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from keras.utils import to_categorical
from keras import layers, models, Model
from keras.applications import EfficientNetB0, ResNet50, VGG16
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras import layers, models, regularizers
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, Dense, Dropout, Flatten
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
# Logging configuration
logging.basicConfig(level=logging.INFO,
                    datefmt='%H:%M:%S',
                    format='%(asctime)s | %(levelname)-5s | %(module)-15s | %(message)s')

IMAGE_SIZE = (299, 299)  # All images contained in this dataset are 299x299 (originally, to match Inception v3 input size)
SEED = 17

# Head directory containing all image subframes. Update with the relative path of your data directory
data_head_dir = Path('data')

# Find all subframe directories
subdirs = [Path(subdir.stem) for subdir in data_head_dir.iterdir() if subdir.is_dir()]
src_image_ids = ['_'.join(a_path.name.split('_')[:3]) for a_path in subdirs]

In [ ]:
# Load train/val/test subframe IDs
def load_text_ids(file_path):
    """Simple helper to load all lines from a text file"""
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines()]
    return lines

# Load the subframe names for the three data subsets
train_ids = load_text_ids('train_source_images.txt')
validate_ids = load_text_ids('val_source_images.txt')
test_ids = load_text_ids('test_source_images.txt')

# Generate a list containing the dataset split for the matching subdirectory names
subdir_splits = []
for src_id in src_image_ids:
    if src_id in train_ids:
        subdir_splits.append('train')
    elif src_id in validate_ids:
        subdir_splits.append('validate')
    elif(src_id in test_ids):
        subdir_splits.append('test')
    else:
        logging.warning(f'{src_id}: Did not find designated split in train/validate/test list.')
        subdir_splits.append(None)

# Loading and pre processing the data

### Note that there are multiple ways to preprocess and load your data in order to train your model in tensorflow. We have provided one way to do it in the following cell. Feel free to use your own method and get better results.


In [ ]:
import random
import tensorflow as tf
from PIL import Image

def load_and_preprocess(img_loc, label):

    def _inner_function(img_loc, label):

        # Convert tensor to native type
        img_loc_str = img_loc.numpy().decode('utf-8')
        label_str = label.numpy().decode('utf-8')

        img = Image.open(img_loc_str).convert('RGB')


        return img, 1 if label_str=='frost' else 0

    # Wrap the Python function
    X, y = tf.py_function(_inner_function, [img_loc, label], [tf.float32, tf.int64])

    return X, y

def load_subdir_data(dir_path, image_size, seed=None):

    """Helper to create a TF dataset from each image subdirectory"""

    # Grab only the classes that (1) we want to keep and (2) exist in this directory
    tile_dir = dir_path / Path('tiles')
    label_dir = dir_path /Path('labels')

    loc_list = []

    for folder in os.listdir(tile_dir):
        if os.path.isdir(os.path.join(tile_dir, folder)):
            for file in os.listdir(os.path.join(tile_dir, folder)):
                if file.endswith(".png"):
                    loc_list.append((os.path.join(os.path.join(tile_dir, folder), file), folder))

    return loc_list

# Loop over all subframes, loading each into a list
tf_data_train, tf_data_test, tf_data_val = [], [], []
tf_dataset_train, tf_dataset_test, tf_dataset_val = [], [], []

# Update the batch and buffer size as per your model requirements
buffer_size = 64
batch_size = 32

for subdir, split in zip(subdirs, subdir_splits):
    full_path = data_head_dir / subdir
    if split=='validate':
        tf_data_val.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))
    elif split=='train':
        tf_data_train.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))
    elif split=='test':
        tf_data_test.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))

random.shuffle(tf_data_train)
img_list, label_list = zip(*tf_data_train)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)


#train dataset
tf_dataset_train = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_train = tf_dataset_train.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_train = tf_dataset_train.shuffle(buffer_size=buffer_size).batch(batch_size)

random.shuffle(tf_data_val)
img_list, label_list = zip(*tf_data_val)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)

tf_dataset_val = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_val = tf_dataset_val.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_val = tf_dataset_val.shuffle(buffer_size=buffer_size).batch(batch_size)

random.shuffle(tf_data_test)
img_list, label_list = zip(*tf_data_train)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)

tf_dataset_test = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_test = tf_dataset_test.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_test = tf_dataset_test.shuffle(buffer_size=buffer_size).batch(batch_size)

# Code begin here please check code below for errors


## image augmentation


In [ ]:
tf_data_train

In [ ]:
print("Train dataset size:", len(tf_dataset_train))
print("Validation dataset size:", len(tf_dataset_val))
print("Test dataset size:", len(tf_dataset_test))

In [ ]:
def augment_image(image, label):
    crop_size = tf.random.uniform(shape=[], minval=250, maxval=299, dtype=tf.int32)
    image = tf.image.random_crop(image, size=[crop_size, crop_size, 3])
    image = tf.image.resize(image, [299, 299])

    zoom_factor = tf.random.uniform(shape=[], minval=0.8, maxval=1.0, dtype=tf.float32)
    image = tf.image.resize(image, [int(299 * zoom_factor), int(299 * zoom_factor)])
    image = tf.image.resize_with_crop_or_pad(image, 299, 299)

    angle = tf.random.uniform(shape=[], minval=-0.2, maxval=0.2, dtype=tf.float32)
    image = tf.image.rot90(image, tf.cast(angle, tf.int32))

    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    image = tf.image.pad_to_bounding_box(image, 20, 20, 339, 339)
    image = tf.image.random_crop(image, size=[299, 299, 3])
    image = tf.image.resize(image, [299, 299])

    return image, label

augmented_train_dataset = tf_dataset_train.map(augment_image, num_parallel_calls=tf.data.experimental.AUTOTUNE)

In [ ]:
augmented_train_dataset

##CNN + MLP

In [ ]:
def cnn_model():
    model = models.Sequential()
    model.add(layers.Conv2D(6, (2, 2), activation='relu', input_shape=(299, 299, 3), kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(3, 3))

    model.add(layers.Conv2D(6, (2, 2), activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(3, 3))

    model.add(layers.Conv2D(6, (2, 2), activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(3, 3))

    model.add(layers.Flatten())
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation='sigmoid'))

    return model

model = cnn_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'], run_eagerly=True)
model.summary()


In [ ]:
epochs = 20
batch_size = 32



# Train the model
history = model.fit(
    tf_dataset_train,
    epochs=epochs,
    validation_data=tf_dataset_val,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    ]
)


In [ ]:
# Plot training and validation errors
plt.plot(history.history['loss'], label='Training Error')
plt.plot(history.history['val_loss'], label='Validation Error')
plt.xlabel('Epochs')
plt.ylabel('Error')
plt.legend()
plt.show()

In [ ]:
# Evaluate the model on the test dataset
predictions = model.predict(tf_dataset_test)
predicted_labels = np.argmax(predictions, axis=1)

# Get true labels from the test dataset
true_labels = np.concatenate([label.numpy() for _, label in tf_dataset_test], axis=0)

# Calculate precision, recall, and F1
precision = precision_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
f1 = f1_score(true_labels, predicted_labels, average='weighted')

# Display the results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Transfer Learning

In [ ]:
from tensorflow.keras.layers.experimental import preprocessing

def load_and_preprocess(img_loc, label):
    def _inner_function(img_loc, label):
        img_loc_str = img_loc.numpy().decode('utf-8')
        label_str = label.numpy().decode('utf-8')

        img = tf.image.decode_png(tf.io.read_file(img_loc_str), channels=3)
        img = tf.image.convert_image_dtype(img, dtype=tf.float32)
        img = tf.image.resize(img, (299, 299))

        return img, 1 if label_str == 'frost' else 0

    X, y = tf.py_function(_inner_function, [img_loc, label], [tf.float32, tf.int64])

    # Set the shape of X and y explicitly
    X.set_shape([299, 299, 3])
    y.set_shape([])

    return X, y

def load_subdir_data(dir_path, image_size, seed=None):

    """Helper to create a TF dataset from each image subdirectory"""

    # Grab only the classes that (1) we want to keep and (2) exist in this directory
    tile_dir = dir_path / Path('tiles')
    label_dir = dir_path /Path('labels')

    loc_list = []

    for folder in os.listdir(tile_dir):
        if os.path.isdir(os.path.join(tile_dir, folder)):
            for file in os.listdir(os.path.join(tile_dir, folder)):
                if file.endswith(".png"):
                    loc_list.append((os.path.join(os.path.join(tile_dir, folder), file), folder))

    return loc_list

# Loop over all subframes, loading each into a list
tf_data_train, tf_data_test, tf_data_val = [], [], []
tf_dataset_train, tf_dataset_test, tf_dataset_val = [], [], []

# Update the batch and buffer size as per your model requirements
buffer_size = 64
batch_size = 32

for subdir, split in zip(subdirs, subdir_splits):
    full_path = data_head_dir / subdir
    if split=='validate':
        tf_data_val.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))
    elif split=='train':
        tf_data_train.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))
    elif split=='test':
        tf_data_test.extend(load_subdir_data(full_path, IMAGE_SIZE, SEED))

random.shuffle(tf_data_train)
img_list, label_list = zip(*tf_data_train)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)


#train dataset
tf_dataset_train = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_train = tf_dataset_train.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_train = tf_dataset_train.shuffle(buffer_size=buffer_size).batch(batch_size)

random.shuffle(tf_data_val)
img_list, label_list = zip(*tf_data_val)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)

tf_dataset_val = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_val = tf_dataset_val.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_val = tf_dataset_val.shuffle(buffer_size=buffer_size).batch(batch_size)

random.shuffle(tf_data_test)
img_list, label_list = zip(*tf_data_train)
img_list_t = tf.convert_to_tensor(img_list)
lb_list_t = tf.convert_to_tensor(label_list)

tf_dataset_test = tf.data.Dataset.from_tensor_slices((img_list_t, lb_list_t))
tf_dataset_test = tf_dataset_test.map(load_and_preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)
tf_dataset_test = tf_dataset_test.shuffle(buffer_size=buffer_size).batch(batch_size)

In [ ]:
# Transfer learning model with ResNet50
base_model_resnet50 = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

for layer in base_model_resnet50.layers:
    layer.trainable = False

model_resnet50 = tf.keras.Sequential([
    base_model_resnet50,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_resnet50.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# Train the ResNet50 model
history_resnet50 = model_resnet50.fit(
    tf_dataset_train,
    epochs=10,
    validation_data=tf_dataset_val,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    ]
)

# Plot training and validation loss for ResNet50
plt.plot(history_resnet50.history['loss'], label='Training Loss (ResNet50)')
plt.plot(history_resnet50.history['val_loss'], label='Validation Loss (ResNet50)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Evaluate the model on the test dataset
predictions = model_resnet50.predict(tf_dataset_test)
predicted_labels = np.argmax(predictions, axis=1)

# Get true labels from the test dataset
true_labels = np.concatenate([label.numpy() for _, label in tf_dataset_test], axis=0)

# Calculate precision, recall, and F1
precision = precision_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
f1 = f1_score(true_labels, predicted_labels, average='weighted')

# Display the results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# Transfer learning model with EfficientNetB0
base_model_efficientnet = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

for layer in base_model_efficientnet.layers:
    layer.trainable = False

model_efficientnet = tf.keras.Sequential([
    base_model_efficientnet,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Use 'sigmoid' for binary classification
])

model_efficientnet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the EfficientNet model
history_efficientnet = model_efficientnet.fit(
    tf_dataset_train,
    epochs=10,
    validation_data=tf_dataset_val,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    ]
)

# Plot training and validation loss for EfficientNet
plt.plot(history_efficientnet.history['loss'], label='Training Loss (EfficientNet)')
plt.plot(history_efficientnet.history['val_loss'], label='Validation Loss (EfficientNet)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
# Evaluate the model on the test dataset
predictions_efficientnet = model_efficientnet.predict(tf_dataset_test)
predicted_labels_efficientnet = (predictions_efficientnet > 0.5).astype(int)

# Get true labels from the test dataset
true_labels_efficientnet = np.concatenate([label.numpy() for _, label in tf_dataset_test], axis=0)

# Calculate precision, recall, and F1
precision_efficientnet = precision_score(true_labels_efficientnet, predicted_labels_efficientnet)
recall_efficientnet = recall_score(true_labels_efficientnet, predicted_labels_efficientnet)
f1_efficientnet = f1_score(true_labels_efficientnet, predicted_labels_efficientnet)

# Display the results
print("Evaluation Metrics for EfficientNet:")
print(f"Precision: {precision_efficientnet:.4f}")
print(f"Recall: {recall_efficientnet:.4f}")
print(f"F1 Score: {f1_efficientnet:.4f}")

In [ ]:
# Transfer learning model with VGG16
base_model_vgg16 = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

for layer in base_model_vgg16.layers:
    layer.trainable = False

model_vgg16 = tf.keras.Sequential([
    base_model_vgg16,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='softmax')
])

model_vgg16.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# Train the VGG16 model
history_vgg16 = model_vgg16.fit(
    tf_dataset_train,
    epochs=10,
    validation_data=tf_dataset_val,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    ]
)

# Plot training and validation loss for VGG16
plt.plot(history_vgg16.history['loss'], label='Training Loss (VGG16)')
plt.plot(history_vgg16.history['val_loss'], label='Validation Loss (VGG16)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Evaluate the model on the test dataset
predictions = model_vgg16.predict(tf_dataset_test)
predicted_labels = np.argmax(predictions, axis=1)

# Get true labels from the test dataset
true_labels = np.concatenate([label.numpy() for _, label in tf_dataset_test], axis=0)

# Calculate precision, recall, and F1
precision = precision_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
f1 = f1_score(true_labels, predicted_labels, average='weighted')

# Display the results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

#EfficientNet:

Precision: 0.5878

Recall: 1.0000

F1 Score: 0.7404

#CNN with MLP (and other architectures - ResNet, VGG16):

Precision: 0.1699

Recall: 0.4122

F1 Score: 0.2407

It's evident that EfficientNet outperforms the CNN with MLP (and other architectures) based on the provided metrics. Here's a breakdown of the comparison:

#Precision:

EfficientNet: 0.5878

CNN with MLP: 0.1699

EfficientNet has a higher precision, indicating that it has a better ability to correctly identify positive instances among the predicted positive instances.

#Recall:

EfficientNet: 1.0000

CNN with MLP: 0.4122

EfficientNet has a perfect recall, meaning it can identify all actual positive instances, while the CNN with MLP has a lower recall.

#F1 Score:

EfficientNet: 0.7404

CNN with MLP: 0.2407

The F1 score, which balances precision and recall, is significantly higher for EfficientNet, indicating better overall performance.
In summary, based on these metrics, it appears that transfer learning using EfficientNet has provided a more effective model compared to training a CNN with MLP (and other architectures) for the given task. EfficientNet has shown higher precision, recall, and F1 score, suggesting superior performance in both identifying positive instances and minimizing false positives.